# Online Appendix: EmbeddingGemma 2 Robustness Check

Writes `Appendix_Gemma.tex`, the section of the Online Appendix that compares the main analysis (fine-tuned DoubleMLDeep embeddings, `main/`) with the re-run using off-the-shelf EmbeddingGemma 2 embeddings (text + image, 256 dimensions, `gemma/`). All tables and numbers are read from the outputs of the executed notebooks:

| content | fine-tuned embeddings | EmbeddingGemma 2 |
| --- | --- | --- |
| model and inference settings | | `gemma/data/embeddings/embeddinggemma2_txtimg.json` |
| clusters, closest products | | `../outputs/01_1_create_dataset_txt_img/cluster_overview.csv`, `../outputs/02_cluster_centroid_products/collage_similarity_cluster_*.png` |
| predictive performance (test $R^2$) | `main/code/03_1_predictive_performance_txt_img.ipynb` | `03_1_predictive_performance_txt_img.ipynb` |
| homogeneous elasticities | `main/output/04_evaluation/homogeneous_elasticity_estimates.csv` | `../outputs/04_evaluation/homogeneous_elasticity_estimates.csv` |
| sensitivity analysis | `main/output/04_evaluation/sensitivity_*.csv` | `../outputs/04_evaluation/sensitivity_*.csv`, `key_results.json` |

The observed-covariate benchmark of the fine-tuned embeddings is taken from the re-run `../outputs/04_evaluation_comparison/finetuned`, which is checked against `main/output/04_evaluation` below.

Requires `04_evaluation.ipynb` (default parameters and the `finetuned` run of `04_evaluation_comparison.ipynb`). Run from `gemma/code`.

In [ ]:
import os
import re
import json
import shutil
from html.parser import HTMLParser
from pathlib import Path

import numpy as np
import pandas as pd
import nbformat

# Parameters
output_dir = "../outputs/05_appendix_gemma"
paper_figure_dir = "paper/figures_revision/gemma"  # figure path used in Appendix_Gemma.tex (Overleaf project)

main_dir = "../../main/output/04_evaluation"
gemma_dir = "../outputs/04_evaluation"
finetuned_rerun_dir = "../outputs/04_evaluation_comparison/finetuned"
Path(output_dir, "figures").mkdir(parents=True, exist_ok=True)

## Inputs and consistency checks

In [ ]:
meta = json.load(open("../data/embeddings/embeddinggemma2_txtimg.json"))

# homogeneous elasticities (Models I-1 to I-3)
ate_main = pd.read_csv(os.path.join(main_dir, "homogeneous_elasticity_estimates.csv"), index_col=0)
ate_gemma = pd.read_csv(os.path.join(gemma_dir, "homogeneous_elasticity_estimates.csv"), index_col=0)
assert list(ate_main.index) == list(ate_gemma.index)

# sensitivity analysis
sens_main = pd.read_csv(os.path.join(main_dir, "sensitivity_calibrated.csv"))
sens_gemma_main_shares = pd.read_csv(os.path.join(gemma_dir, "sensitivity_main_shares.csv"))
sens_gemma_own = pd.read_csv(os.path.join(gemma_dir, "sensitivity_calibrated.csv"))
rv_main = pd.read_csv(os.path.join(main_dir, "sensitivity_robustness_values.csv")).set_index("null_hypothesis")
rv_gemma = pd.read_csv(os.path.join(gemma_dir, "sensitivity_robustness_values.csv")).set_index("null_hypothesis")
assert np.allclose(sens_gemma_main_shares["share"], sens_main["share"], rtol=0, atol=0)
key_gemma = json.load(open(os.path.join(gemma_dir, "key_results.json")))["sensitivity"]
key_main = json.load(open(os.path.join(finetuned_rerun_dir, "key_results.json")))["sensitivity"]

# the re-run of the fine-tuned specification reproduces main/output/04_evaluation
for file_name in ["homogeneous_elasticity_estimates.csv", "sensitivity_calibrated.csv", "sensitivity_robustness_values.csv"]:
    a = pd.read_csv(os.path.join(main_dir, file_name)).select_dtypes("number")
    b = pd.read_csv(os.path.join(finetuned_rerun_dir, file_name)).select_dtypes("number")
    assert np.allclose(a.values, b.values, rtol=0, atol=1e-9), file_name
assert np.isclose(key_main["scaled_implied_rv"], sens_main.loc[0, "share"], rtol=0, atol=1e-12)
assert np.isclose(key_gemma["scaled_implied_rv"], sens_gemma_own.loc[0, "share"], rtol=0, atol=1e-12)
print("fine-tuned re-run reproduces main/output/04_evaluation")

In [ ]:
class _TableParser(HTMLParser):
    """Minimal parser for the HTML representation of a pandas DataFrame."""

    def __init__(self):
        super().__init__()
        self.rows, self._row, self._cell = [], None, None

    def handle_starttag(self, tag, attrs):
        if tag == "tr":
            self._row = []
        elif tag in ("td", "th") and self._row is not None:
            self._cell = ""

    def handle_endtag(self, tag):
        if tag in ("td", "th") and self._cell is not None:
            self._row.append(self._cell.strip())
            self._cell = None
        elif tag == "tr" and self._row is not None:
            self.rows.append(self._row)
            self._row = None

    def handle_data(self, data):
        if self._cell is not None:
            self._cell += data


def read_result_table(notebook, variable):
    """Read the DataFrame displayed by the cell `<variable> = pd.concat(...)` of an executed notebook."""
    nb = nbformat.read(notebook, as_version=4)
    cells = [c for c in nb.cells if c.cell_type == "code" and re.match(rf"{variable} = pd\.concat", c.source.strip())]
    assert len(cells) == 1, (notebook, variable)
    html = [o["data"]["text/html"] for o in cells[0].outputs if "data" in o and "text/html" in o["data"]]
    assert len(html) == 1
    parser = _TableParser()
    parser.feed(html[0])
    header, *body = parser.rows
    df = pd.DataFrame([r[1:] for r in body], index=[r[0] for r in body], columns=header[1:]).astype(float)
    return df


r2 = {}
for label, notebook in [
    ("finetuned", "../../main/code/03_1_predictive_performance_txt_img.ipynb"),
    ("gemma", "03_1_predictive_performance_txt_img.ipynb"),
]:
    levels = read_result_table(notebook, "results_df")
    diffs = read_result_table(notebook, "results_df_diff")
    r2[label] = pd.DataFrame(
        {
            "Q": levels["R2 Q Test"],
            "P": levels["R2 P Test"],
            "dQ": diffs["R2 Q Test"],
            "dP": diffs["R2 P Test"],
        }
    )
# the tabular-only benchmarks do not depend on the embeddings
for row in ["OLS (Tabular)", "Boosting (Tabular)"]:
    assert np.allclose(r2["finetuned"].loc[row], r2["gemma"].loc[row], rtol=0, atol=1e-12)
pd.concat(r2, axis=1).round(4)

## Tables

In [ ]:
def num(x, digits=3):
    """Number in math mode (proper minus sign)."""
    return f"${x:.{digits}f}$"


def pct(x, digits=2):
    """Test R^2 in percent; negative values are set to 0 as in the main text."""
    return f"{100 * max(x, 0.0):.{digits}f}"


# --- predictive performance -----------------------------------------------------------
r2_rows = [
    ("Linear Reg [all tabular]", "OLS (Tabular)"),
    ("Boosted Trees Reg [all tabular]", "Boosting (Tabular)"),
    None,
    ("Linear Reg [+5 PCAs]", "OLS (Tabular + PCA)"),
    ("Linear Reg [+5 Similarities]", "OLS (Tabular + Similarities)"),
    ("Linear Reg [+256 Embeddings]", "OLS (Tabular + Embeddings)"),
    ("Boosted Trees Reg [+5 PCAs]", "Boosting (Tabular + PCA)"),
    ("Boosted Trees Reg [+5 Similarities]", "Boosting (Tabular + Similarities)"),
    ("Boosted Trees Reg [+256 Embeddings]", "Boosting (Tabular + Embeddings)"),
]
lines = []
for row in r2_rows:
    if row is None:
        lines.append("\\midrule")
        continue
    label, key = row
    cells = []
    for target in ["Q", "P", "dQ", "dP"]:
        a, b = r2["finetuned"].loc[key, target], r2["gemma"].loc[key, target]
        if key in ("OLS (Tabular)", "Boosting (Tabular)"):
            cells.append(f"\\multicolumn{{2}}{{c}}{{{pct(a)}}}")
        else:
            cells.append(f"{pct(a)} & {pct(b)}")
    lines.append(f"{label} & " + " & ".join(cells) + "\\\\")
table_r2 = "\n".join(lines)

# --- homogeneous elasticities -----------------------------------------------------------
lines = []
for spec in ate_main.index:
    cells = []
    for df in [ate_main, ate_gemma]:
        r = df.loc[spec]
        cells.append(f"{num(r['coef'])} & {num(r['std err'])} & [{num(r['lower'])}, {num(r['upper'])}]")
    lines.append(f"{{{spec}}} & " + " & ".join(cells) + "\\\\")
table_ate = "\n".join(lines)


# --- sensitivity at the residual shares of the main analysis ----------------------------
def sensitivity_rows(df):
    return "\n".join(
        f"{r['scenario']} & {100 * r['share']:.2f}\\% & {num(r['theta_lower'])} & {num(r['theta_upper'])} "
        f"& {num(r['ci_lower'])} & {num(r['ci_upper'])}\\\\"
        for _, r in df.iterrows()
    )


table_sens_main = sensitivity_rows(sens_main)
table_sens_gemma = sensitivity_rows(sens_gemma_main_shares)

# --- benchmark and robustness values ------------------------------------------------------
bench_rows = [
    ("$\\tilde R_Y^2$ (omitting $X^{o}_t$)", key_main["partial_r2_outcome"], key_gemma["partial_r2_outcome"], 4),
    ("$\\tilde R_D^2$ (omitting $X^{o}_t$)", key_main["partial_r2_treatment"], key_gemma["partial_r2_treatment"], 4),
    ("$|\\tilde\\rho|$", key_main["implied_rho"], key_gemma["implied_rho"], 4),
    ("Calibration $r_0$", key_main["scaled_implied_rv"], key_gemma["scaled_implied_rv"], 4),
]
lines = [f"{label} & {a:.{d}f} & {b:.{d}f}\\\\" for label, a, b, d in bench_rows]
lines.append("\\midrule")
for null, label in [(0.0, "0"), (-0.5, "-0.5")]:
    lines.append(
        f"RV, $H_0\\colon \\alpha={label}$ & {100 * rv_main.loc[null, 'RV']:.2f}\\% & {100 * rv_gemma.loc[null, 'RV']:.2f}\\%\\\\"
    )
    lines.append(
        f"RVa, $H_0\\colon \\alpha={label}$ & {100 * rv_main.loc[null, 'RVa']:.2f}\\% & {100 * rv_gemma.loc[null, 'RVa']:.2f}\\%\\\\"
    )
table_bench = "\n".join(lines)
print(table_r2, table_ate, table_sens_main, table_sens_gemma, table_bench, sep="\n\n")

## Appendix_Gemma.tex

Text of the section (`templates/Appendix_Gemma.tex` with placeholders `<<name>>`). Qualitative statements of the text are checked against the estimates.

In [ ]:
from sklearn.metrics import adjusted_rand_score


def f3(x):
    return f"{x:.3f}"


def m3(x):
    """Number with three decimals in math mode."""
    return f"${x:.3f}$"


def p2(x):
    """Share in percent with two decimals."""
    return f"{100 * x:.2f}\\%"


# --- clusters -----------------------------------------------------------------------------
sim_cols = [f"similarity_cluster_{k}" for k in range(5)]
assignments = {}
for label, data_dir in [("finetuned", "../../main/data"), ("gemma", "../data")]:
    frames = [
        pd.read_csv(f"{data_dir}/dataset_txt_only_False_embeddings_True_{split}.zip", usecols=["ASIN", "date"] + sim_cols)
        for split in ["train", "val"]
    ]
    df = pd.concat(frames, ignore_index=True)
    assignments[label] = df
assert assignments["finetuned"][["ASIN", "date"]].equals(assignments["gemma"][["ASIN", "date"]])
ari = adjusted_rand_score(
    assignments["finetuned"][sim_cols].values.argmax(axis=1), assignments["gemma"][sim_cols].values.argmax(axis=1)
)

overview = pd.read_csv("../outputs/01_1_create_dataset_txt_img/cluster_overview.csv")


def top_share(embedding):
    """Share of the most frequent product subcategory in each cluster (from cluster_overview.csv)."""
    rows = overview[overview["embedding"] == embedding].set_index("cluster")["Top product types"]
    return rows.map(lambda s: int(re.match(r"(.+?) \((\d+)%\)", s).group(2)) / 100), rows.map(
        lambda s: re.match(r"(.+?) \((\d+)%\)", s).group(1)
    )


share_ft, type_ft = top_share("finetuned")
share_eg, type_eg = top_share("gemma_txtimg_dim256")
n_products_eg = overview[overview["embedding"] == "gemma_txtimg_dim256"].set_index("cluster")["# Products"]
print(f"ARI = {ari:.4f}")
print(pd.DataFrame({"top type FT": type_ft, "share FT": share_ft, "top type EG": type_eg, "share EG": share_eg, "# products EG": n_products_eg}))

# statements about the clusters
assert ari < 0.1
assert type_eg[2] == "Skateboards" and share_eg[2] >= 0.85
assert type_eg[3] == "Train Sets"
assert share_eg.min() >= share_ft.min() and share_eg.max() > share_ft.max()
assert (type_ft == "Cars & Race Cars").all()
assert list(type_eg) == ["Trucks", "Cars & Race Cars", "Skateboards", "Train Sets", "Cars & Race Cars"]

cluster_text = (
    "\\revised{Figure~\\ref{fig:gemma_clusters} shows the four products closest to each of the five cluster centers "
    "of the EmbeddingGemma 2 embeddings, computed with Algorithm~\\ref{alg:features}. These clusters define the "
    "similarity features $X^{sim}_i$ in the estimations below. The products closest to the centers are toddler "
    "construction-truck sets (cluster 0), 1:64 scale die-cast cars (cluster 1), fingerboards (cluster 2), toy train "
    "sets (cluster 3), and larger-scale die-cast model cars (cluster 4). The clusters are more closely aligned with "
    f"product categories than those of the fine-tuned embeddings. In cluster 2, {100 * share_eg[2]:.0f}\\% of the "
    "products belong to the subcategory of skateboards, which contains the fingerboards, and the most frequent "
    "subcategories of the five clusters are trucks, cars and race cars, skateboards, train sets, and again cars and "
    "race cars. For the fine-tuned embeddings, the subcategory of cars and race cars is the "
    "most frequent one in all five clusters, with shares between "
    f"{100 * share_ft.min():.0f}\\% and {100 * share_ft.max():.0f}\\%. The two cluster assignments are almost "
    f"unrelated (adjusted Rand index {ari:.2f}). This is consistent with fine-tuning organizing the embedding space "
    "around characteristics that predict prices and quantities rather than around generic product categories.}"
)

cluster_figure_rows = "\n".join(
    f"Cluster {k} & \\includegraphics[width=.8\\textwidth]{{{paper_figure_dir}/collage_similarity_cluster_{k}.png}}\\\\[0.1cm]"
    for k in range(5)
)

# --- predictive performance -------------------------------------------------------------------
ft, eg = r2["finetuned"], r2["gemma"]
q_emb_ft, q_emb_eg = ft.loc["Boosting (Tabular + Embeddings)", "Q"], eg.loc["Boosting (Tabular + Embeddings)", "Q"]
p_emb_ft, p_emb_eg = ft.loc["Boosting (Tabular + Embeddings)", "P"], eg.loc["Boosting (Tabular + Embeddings)", "P"]
dq_emb_ft, dq_emb_eg = ft.loc["Boosting (Tabular + Embeddings)", "dQ"], eg.loc["Boosting (Tabular + Embeddings)", "dQ"]
p_tab = ft.loc["Boosting (Tabular)", "P"]
p_lin_tab = ft.loc["OLS (Tabular)", "P"]
q_lin_tab = ft.loc["OLS (Tabular)", "Q"]
compressed = ["Boosting (Tabular + PCA)", "Boosting (Tabular + Similarities)"]
compressed_lin = ["OLS (Tabular + PCA)", "OLS (Tabular + Similarities)"]
dp_max = max(ft["dP"].max(), eg["dP"].max())

assert q_emb_eg > q_emb_ft and dq_emb_eg > dq_emb_ft and p_emb_eg < p_emb_ft
assert (ft.loc[compressed, "P"] > 0.6).all() and (eg.loc[compressed, "P"] < 0.4).all()
assert (eg.loc[compressed_lin, "P"] - p_lin_tab).abs().max() < 0.015
assert (eg.loc[compressed_lin, "Q"] > q_lin_tab + 0.05).all() and (eg.loc[compressed_lin, "Q"] < ft.loc[compressed_lin, "Q"]).all()
assert dp_max < 0.02

r2_text = (
    "\\revised{Table~\\ref{tab:gemma_r2} compares the test $R^2$ of the prediction models of the main text when the "
    "embedding features are derived from EmbeddingGemma 2. With all 256 coordinates, boosted trees predict quantity "
    f"levels better with EmbeddingGemma 2 than with the fine-tuned embeddings ({p2(q_emb_eg)} vs.\\ {p2(q_emb_ft)}), "
    f"and changes in quantity as well ({p2(dq_emb_eg)} vs.\\ {p2(dq_emb_ft)}), but price levels worse "
    f"({p2(p_emb_eg)} vs.\\ {p2(p_emb_ft)}). The compressed features differ more. For the fine-tuned embeddings, "
    "five principal components or five cluster similarities retain most of the predictive information on prices "
    f"({p2(ft.loc[compressed[0], 'P'])} and {p2(ft.loc[compressed[1], 'P'])} with boosted trees). The corresponding "
    f"EmbeddingGemma 2 features reach {p2(eg.loc[compressed[0], 'P'])} and {p2(eg.loc[compressed[1], 'P'])}, compared "
    f"with {p2(p_tab)} for the tabular controls alone. In linear regressions, they raise the $R^2$ for quantity "
    f"from {p2(q_lin_tab)} to {p2(eg.loc[compressed_lin[0], 'Q'])} and {p2(eg.loc[compressed_lin[1], 'Q'])} "
    f"and leave the $R^2$ for prices almost unchanged ({p2(eg.loc[compressed_lin[0], 'P'])} and "
    f"{p2(eg.loc[compressed_lin[1], 'P'])} vs.\\ {p2(p_lin_tab)}). Fine-tuning thus concentrates the price-relevant "
    "information in a few directions of the embedding space; without fine-tuning, it is spread over the full "
    f"embedding. Changes in prices remain essentially unpredictable with both embeddings (at most {p2(dp_max)}).}}"
)

# --- homogeneous elasticities -----------------------------------------------------------------
specs_emb = [s for s in ate_main.index if "X^{e}" in s or "X^{sim}" in s]
diff = (ate_gemma.loc[specs_emb, "coef"] - ate_main.loc[specs_emb, "coef"]).abs()
diff_in_se = diff / ate_main.loc[specs_emb, "std err"]
spec_e = "I-3. Boosted Trees ($P_{t-1}$, $Q_{t-1}$, $X^{e}, X^{o}_t$)"
assert spec_e in specs_emb
linear_specs = [s for s in specs_emb if s.startswith(("I-1", "I-2"))]

assert diff_in_se.max() < 0.5
assert (ate_gemma["upper"] < -0.6).all() and (ate_main["upper"] < -0.6).all()
assert (ate_gemma.loc[linear_specs, "coef"] < ate_main.loc[linear_specs, "coef"]).all()
assert np.isclose(ate_gemma.iloc[0]["coef"], ate_main.iloc[0]["coef"], rtol=0, atol=1e-12)

ate_text = (
    "\\revised{Table~\\ref{tab:gemma_ate} reports the homogeneous elasticity estimates of the main text for both "
    "embeddings. In the specifications with embedding features ($X^{e}$ or $X^{sim}$), the EmbeddingGemma 2 estimates "
    f"range from {m3(ate_gemma.loc[specs_emb, 'coef'].min())} to {m3(ate_gemma.loc[specs_emb, 'coef'].max())}, "
    f"compared with {m3(ate_main.loc[specs_emb, 'coef'].min())} to {m3(ate_main.loc[specs_emb, 'coef'].max())} for "
    f"the fine-tuned embeddings. The largest difference between point estimates is {f3(diff.max())}, less than half "
    "a standard error. The linear specifications give slightly more negative estimates with EmbeddingGemma 2. For the "
    f"boosted-tree specification with embeddings (Model I-3), the estimate is {m3(ate_gemma.loc[spec_e, 'coef'])} "
    f"(standard error {f3(ate_gemma.loc[spec_e, 'std err'])}), compared with {m3(ate_main.loc[spec_e, 'coef'])} "
    f"({f3(ate_main.loc[spec_e, 'std err'])}). All upper endpoints of the 90\\% intervals lie below $-0.6$ for both "
    "embeddings, so the finding of elastic demand does not depend on fine-tuning the embeddings.}"
)

# --- sensitivity ------------------------------------------------------------------------------
ci_eg, ci_ft = sens_gemma_main_shares["ci_upper"], sens_main["ci_upper"]
own_ci = sens_gemma_own["ci_upper"]
assert (ci_eg < 0).all() and (ci_ft < 0).all()
assert (ci_eg.iloc[:2] < -0.5).all() and (ci_ft.iloc[:2] < -0.5).all()
assert ci_eg.iloc[2] < -0.5 < ci_ft.iloc[2] and ci_ft.iloc[2] - ci_eg.iloc[2] < 0.01
assert (sens_gemma_main_shares["theta_upper"] < -0.5).all()
assert key_gemma["implied_rho"] < key_main["implied_rho"]
assert key_gemma["scaled_implied_rv"] < key_main["scaled_implied_rv"]
assert (own_ci < -0.5).all()
assert abs(key_gemma["partial_r2_outcome"] - key_main["partial_r2_outcome"]) < 0.01

sensitivity_text = (
    "\\revised{Table~\\ref{tab:gemma_sensitivity} evaluates Model I-3 with embeddings at the residual shares of the "
    "calibrated scenarios of Table~\\ref{tab:sensitivity_calibrated}, so that the omitted confounding has the same "
    "strength for both embeddings. The bounds are very similar. With EmbeddingGemma 2, the upper confidence bounds are "
    f"{m3(ci_eg[0])}, {m3(ci_eg[1])}, and {m3(ci_eg[2])}, compared with {m3(ci_ft[0])}, {m3(ci_ft[1])}, and "
    f"{m3(ci_ft[2])} for the fine-tuned embeddings. A negative effect is robust in all three scenarios; the upper "
    f"confidence bound reaches zero only at a share of {p2(rv_gemma.loc[0.0, 'RVa'])} "
    f"({p2(rv_main.loc[0.0, 'RVa'])} for the fine-tuned embeddings). Elastic demand is robust at the calibration and "
    "its 1.5 multiple for both embeddings. At twice the calibration, the upper confidence bound lies marginally below "
    "$-0.5$ with EmbeddingGemma 2 and marginally above it with the fine-tuned embeddings: it reaches $-0.5$ at a share "
    f"of {p2(rv_gemma.loc[-0.5, 'RVa'])}, compared with {p2(rv_main.loc[-0.5, 'RVa'])}.}}\n\n"
    "\\revised{Table~\\ref{tab:gemma_benchmark} reports the observed-covariate benchmark. Omitting the tabular controls "
    f"gives similar partial $R^2$ values for EmbeddingGemma 2 ($\\tilde R_Y^2={key_gemma['partial_r2_outcome']:.4f}$, "
    f"$\\tilde R_D^2={key_gemma['partial_r2_treatment']:.4f}$), but a weaker alignment "
    f"($|\\tilde\\rho|={key_gemma['implied_rho']:.4f}$ vs.\\ ${key_main['implied_rho']:.4f}$). The calibration "
    f"based on the EmbeddingGemma 2 benchmark is therefore $r_0={100 * key_gemma['scaled_implied_rv']:.2f}\\%$, "
    f"smaller than the {100 * key_main['scaled_implied_rv']:.2f}\\% of the main analysis. At 1, 1.5, and 2 times this "
    f"calibration, the upper confidence bounds are {m3(own_ci[0])}, {m3(own_ci[1])}, and {m3(own_ci[2])}, all below "
    "$-0.5$. We report the comparison at the calibration of the main analysis because it holds the strength of "
    "omitted confounding fixed across embeddings.}"
)

all_eg = ate_gemma["coef"]
conclusion_text = (
    "\\revised{Overall, off-the-shelf EmbeddingGemma 2 embeddings lead to the same conclusions as the fine-tuned "
    f"embeddings: elastic demand, with homogeneous inverse-rank elasticities between {m3(all_eg.min())} and "
    f"{m3(all_eg.max())}, and a similar robustness to omitted confounding. Fine-tuning mainly matters for "
    "prediction and for the interpretation of the clusters: it concentrates the price-relevant information in a few "
    "principal components or cluster similarities, and it yields clusters that cut across product categories. "
    "The elasticity estimates, which control for the embedding features together with the lagged quantity and price "
    "signals and the tabular controls, are insensitive to this choice.}"
)

FILL = {
    "revision": meta["model_revision"][:7],
    "vision_tokens": str(meta["vision_soft_tokens_per_image"]),
    "cluster_text": cluster_text,
    "cluster_figure_rows": cluster_figure_rows,
    "r2_text": r2_text,
    "table_r2": table_r2,
    "ate_text": ate_text,
    "table_ate": table_ate,
    "sensitivity_text": sensitivity_text,
    "alpha_main": f3(sens_main.loc[0, "theta"]),
    "alpha_gemma": f3(sens_gemma_main_shares.loc[0, "theta"]),
    "table_sens_main": table_sens_main,
    "table_sens_gemma": table_sens_gemma,
    "r0_main_7": f"{key_main['scaled_implied_rv']:.7f}",
    "table_bench": table_bench,
    "conclusion_text": conclusion_text,
}
assert np.isclose(sens_main.loc[0, "theta"], ate_main.loc[spec_e, "coef"], rtol=0, atol=1e-12)
assert np.isclose(sens_gemma_main_shares.loc[0, "theta"], ate_gemma.loc[spec_e, "coef"], rtol=0, atol=1e-12)


In [ ]:
with open("templates/Appendix_Gemma.tex") as f:
    template = f.read()
template = template.split("\n", 2)[2]  # drop the template comment lines

text = template
for key, value in FILL.items():
    placeholder = "<<" + key + ">>"
    assert text.count(placeholder) >= 1, key
    text = text.replace(placeholder, value)
assert not re.findall(r"<<\w+>>", text), re.findall(r"<<\w+>>", text)

with open(os.path.join(output_dir, "Appendix_Gemma.tex"), "w") as f:
    f.write(text)
for k in range(5):
    shutil.copy(
        f"../outputs/02_cluster_centroid_products/collage_similarity_cluster_{k}.png",
        os.path.join(output_dir, "figures", f"collage_similarity_cluster_{k}.png"),
    )
print(text)